# Cleaning FMR Data

**Goal:**  
Prepare the Fair Market Rent data so it can be used later in the New Jersey housing affordability analysis.

**Plan:**

1. Load the raw FMR file
2. Filter the data to New Jersey
3. Reshape the data from wide format to county-year format
4. Remove unneeded location columns
5. Fix data types
6. Check for missing values and duplicate county-year rows
7. Save the cleaned FMR dataset

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

### Step 1: Load the Raw FMR File


In [ ]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

fmr_path = project_root / "data" / "FMR" / "FMR_All_1983_2026.csv"
clean_path = project_root / "clean_data"

clean_path.mkdir(exist_ok=True)

fmr_raw = pd.read_csv(fmr_path, dtype=str, encoding="latin1")

fmr_raw.head()

### Step 2: Filter to New Jersey Counties

In [ ]:
fmr_nj = fmr_raw[fmr_raw["state"].str.zfill(2) == "34"].copy()
fmr_nj.head()

In [ ]:
fmr_nj.shape

The data only has 21 columns right now because it represents the 21 counties, the years are specified in the columns.

### Step 3: Reshape the FMR Data

Every Row must be a Unique county and year.

In [ ]:
years = range(1993, 2014)

id_cols = [
    "state",
    "county",
    "cousub",
    "name",
    "fips",
    "fips2024"
]

fmr_yearly_dfs = []

for year in years:
    yy = str(year)[-2:]
    
    year_cols = [
        f"fmr{yy}_0",
        f"fmr{yy}_1",
        f"fmr{yy}_2",
        f"fmr{yy}_3",
        f"fmr{yy}_4",
        f"fmr{yy}"
    ]
    
    temp = fmr_nj[id_cols + year_cols].copy()
    temp["year"] = year
    
    temp = temp.rename(columns={
        f"fmr{yy}_0": "fmr_0br",
        f"fmr{yy}_1": "fmr_1br",
        f"fmr{yy}_2": "fmr_2br",
        f"fmr{yy}_3": "fmr_3br",
        f"fmr{yy}_4": "fmr_4br",
        f"fmr{yy}": "fmr_percentile"
    })
    
    fmr_yearly_dfs.append(temp)

fmr = pd.concat(fmr_yearly_dfs, ignore_index=True)

fmr.head()

In [ ]:
fmr.shape

### Step 4: Remove Unneeded Columns

Some columns are extra location details from the original FMR file. The only location needed is county.

- `state`: all rows are New Jersey
- `county`: only a county code, not the county name
- `cousub`: extra county subdivision detail
- `fips`: extra location detail
- `fips2024`: extra location detail

In [ ]:
# Create the 5-digit county FIPS code using state + county code
fmr["county_fips"] = (
    fmr["state"].astype(str).str.zfill(2) +
    fmr["county"].astype(str).str.zfill(3)
)

# Replace the county code with the county name
fmr["county"] = (
    fmr["name"]
    .str.replace(" County", "", regex=False)
    .str.strip()
)

# Remove extra location columns
fmr = fmr.drop(columns=["state", "cousub", "name", "fips", "fips2024"])

# Put columns in the order I want
fmr = fmr[[
    "county_fips",
    "county",
    "year",
    "fmr_0br",
    "fmr_1br",
    "fmr_2br",
    "fmr_3br",
    "fmr_4br",
    "fmr_percentile"
]]

fmr.head()

### Step 5: Fix Data Types

In [ ]:
fmr.info()

The rent columns are showing as `object`, which means pandas is reading them as text. Since these columns are rent values, they need to be changed to numbers. The `county_fips` column can stay as text because it is an ID, not a value for math.

In [ ]:
rent_cols = [
    "fmr_0br",
    "fmr_1br",
    "fmr_2br",
    "fmr_3br",
    "fmr_4br",
    "fmr_percentile"
]

for col in rent_cols:
    fmr[col] = pd.to_numeric(fmr[col], errors="coerce")

fmr.info()

### Step 6: Check for Missing Values and Duplicates


In [ ]:
fmr.isnull().sum()

In [ ]:
fmr.duplicated(subset=["county_fips", "year"]).sum()

In [ ]:
fmr["year"].value_counts().sort_index()

### Step 7: Save the Cleaned FMR Data

In [ ]:
fmr.to_csv(clean_path / "FMR_clean.csv", index=False)

In [ ]:
fmr.head()